# 02 — OGG Phase 1 Exploratory Data Analysis

This notebook characterizes the downloaded OGG flight, NOAA airport-weather, and Hawaii Storm Events datasets before we construct the time-aligned modeling table.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if (cwd / 'data').exists():
    ROOT = cwd
elif (cwd.parent / 'data').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(f'Cannot locate project root from {cwd}')

BTS_FILE = ROOT / 'data/raw/bts/ogg_flights_2020_2026.csv.gz'
WEATHER_FILE = ROOT / 'data/raw/weather/ogg_lcd_2020_2026.csv.gz'
STORM_FILE = ROOT / 'data/raw/incidents/hawaii_storm_events_2020_2026.csv.gz'

print('ROOT:', ROOT)
print('Flights exists:', BTS_FILE.exists())
print('Weather exists:', WEATHER_FILE.exists())
print('Storms exists:', STORM_FILE.exists())

for p in [BTS_FILE, WEATHER_FILE, STORM_FILE]:
    if not p.exists():
        raise FileNotFoundError(p)


## 1. Flights: load, normalize, label, and inspect


In [ ]:
flights = pd.read_csv(BTS_FILE, low_memory=False)
print('Raw flight columns:', list(flights.columns))

# The BTS downloader stores delay variables as *DelayMinutes.
# Create stable aliases used throughout the analysis.
if 'ArrDelay' not in flights.columns and 'ArrDelayMinutes' in flights.columns:
    flights['ArrDelay'] = pd.to_numeric(flights['ArrDelayMinutes'], errors='coerce')
if 'DepDelay' not in flights.columns and 'DepDelayMinutes' in flights.columns:
    flights['DepDelay'] = pd.to_numeric(flights['DepDelayMinutes'], errors='coerce')

for col in ['Cancelled', 'Diverted', 'ArrDelay', 'DepDelay']:
    if col in flights.columns:
        flights[col] = pd.to_numeric(flights[col], errors='coerce')

required = ['FlightDate', 'Origin', 'Dest', 'Cancelled']
missing_required = [c for c in required if c not in flights.columns]
if missing_required:
    raise KeyError(f'Missing required BTS columns: {missing_required}')

flights['FlightDate'] = pd.to_datetime(flights['FlightDate'], errors='coerce')
flights['year'] = flights['FlightDate'].dt.year
flights['month'] = flights['FlightDate'].dt.month
flights['year_month'] = flights['FlightDate'].dt.to_period('M').astype(str)
flights['direction'] = np.where(flights['Origin'].eq('OGG'), 'departure', 'arrival')
flights['covid_era'] = flights['FlightDate'].between('2020-03-01', '2021-05-31')

def disruption_class(r):
    if r.get('Cancelled', 0) == 1:
        return 'cancelled'
    d = r.get('ArrDelay', np.nan)
    if pd.isna(d):
        return 'unknown'
    if d < 15:
        return 'normal'
    if d < 180:
        return 'delay'
    return 'severe_delay'

flights['disruption_class'] = flights.apply(disruption_class, axis=1)

print('Flights:', flights.shape)
display(flights.head())
display(flights['disruption_class'].value_counts(dropna=False).to_frame('count'))
display((flights['disruption_class'].value_counts(normalize=True) * 100).round(2).to_frame('percent'))


In [ ]:
yearly_aggs = {
    'flights': ('FlightDate', 'size'),
    'cancelled': ('Cancelled', 'sum'),
}
if 'ArrDelay' in flights.columns:
    yearly_aggs['mean_arr_delay'] = ('ArrDelay', 'mean')
    yearly_aggs['median_arr_delay'] = ('ArrDelay', 'median')

yearly = flights.groupby('year').agg(**yearly_aggs)
yearly['cancel_rate_pct'] = 100 * yearly['cancelled'] / yearly['flights']
display(yearly.round(2))

carrier_col = 'Reporting_Airline' if 'Reporting_Airline' in flights.columns else None
if carrier_col:
    airline_aggs = {
        'flights': ('FlightDate', 'size'),
        'cancelled': ('Cancelled', 'sum'),
        'severe_delays': ('disruption_class', lambda s: (s == 'severe_delay').sum()),
    }
    if 'ArrDelay' in flights.columns:
        airline_aggs['mean_arr_delay'] = ('ArrDelay', 'mean')
    airlines = flights.groupby(carrier_col).agg(**airline_aggs)
    airlines['cancel_rate_pct'] = 100 * airlines['cancelled'] / airlines['flights']
    airlines['severe_delay_rate_pct'] = 100 * airlines['severe_delays'] / airlines['flights']
    display(airlines.sort_values('flights', ascending=False).round(2))
else:
    print('Reporting_Airline column not found; airline summary skipped.')

display(pd.crosstab(
    flights['direction'],
    flights['disruption_class'],
    normalize='index'
).mul(100).round(2))

covid_aggs = {
    'flights': ('FlightDate', 'size'),
    'cancelled': ('Cancelled', 'sum'),
}
if 'ArrDelay' in flights.columns:
    covid_aggs['mean_arr_delay'] = ('ArrDelay', 'mean')
display(flights.groupby('covid_era').agg(**covid_aggs).round(2))


In [ ]:
monthly_aggs = {
    'flights': ('FlightDate', 'size'),
    'cancelled': ('Cancelled', 'sum'),
}
if 'ArrDelay' in flights.columns:
    monthly_aggs['mean_arr_delay'] = ('ArrDelay', 'mean')

monthly = flights.groupby('year_month').agg(**monthly_aggs)
monthly['cancel_rate_pct'] = 100 * monthly['cancelled'] / monthly['flights']
display(monthly.tail(30).round(2))

flights['disruption_class'].value_counts().plot(
    kind='bar', figsize=(8, 4), title='OGG disruption classes'
)
plt.ylabel('Flights')
plt.tight_layout()
plt.show()

yearly['cancel_rate_pct'].plot(
    marker='o', figsize=(8, 4), title='OGG cancellation rate by year'
)
plt.ylabel('Cancellation rate (%)')
plt.tight_layout()
plt.show()


## 2. NOAA airport weather: schema and missingness


In [ ]:
weather = pd.read_csv(WEATHER_FILE, low_memory=False)
print('Weather:', weather.shape)
display(weather.head())
display(pd.DataFrame({'column': weather.columns}))

missing = (
    weather.isna().mean().mul(100)
    .sort_values(ascending=False)
    .rename('missing_pct')
    .to_frame()
)
display(missing.head(30).round(2))

keywords = [
    'date', 'hourly', 'wind', 'gust', 'visibility', 'precip',
    'pressure', 'temperature', 'dew', 'sky', 'weather'
]
candidate_cols = [
    c for c in weather.columns
    if any(k in c.lower() for k in keywords)
]
display(pd.DataFrame({'candidate_weather_feature': candidate_cols}))


## 3. NOAA Hawaii Storm Events: event types and timing


In [ ]:
storms = pd.read_csv(STORM_FILE, low_memory=False)
print('Storm events:', storms.shape)
display(storms.head())

if 'EVENT_TYPE' in storms.columns:
    display(storms['EVENT_TYPE'].value_counts().head(30).to_frame('count'))
else:
    print('EVENT_TYPE not found.')

if {'YEAR', 'EVENT_TYPE'}.issubset(storms.columns):
    display(pd.crosstab(storms['YEAR'], storms['EVENT_TYPE']))

useful = [
    c for c in [
        'EVENT_ID', 'EVENT_TYPE', 'BEGIN_DATE_TIME', 'END_DATE_TIME',
        'CZ_NAME', 'SOURCE', 'MAGNITUDE', 'DEATHS_DIRECT',
        'INJURIES_DIRECT', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS',
        'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON',
        'EVENT_NARRATIVE'
    ]
    if c in storms.columns
]
display(storms[useful].head(20))


## 4. Broad storm-day overlap check

This is intentionally broad: **any Hawaii Storm Event active on the same calendar day**. Notebook 03 will refine this to Maui/OGG-specific spatial and hourly matching.


In [ ]:
if {'BEGIN_DATE_TIME', 'END_DATE_TIME'}.issubset(storms.columns):
    storms['begin_dt'] = pd.to_datetime(
        storms['BEGIN_DATE_TIME'], errors='coerce'
    )
    storms['end_dt'] = pd.to_datetime(
        storms['END_DATE_TIME'], errors='coerce'
    )

    valid = storms[['begin_dt', 'end_dt']].dropna()
    valid = valid[valid['end_dt'] >= valid['begin_dt']]

    ranges = []
    for r in valid.itertuples(index=False):
        ranges.extend(
            pd.date_range(
                r.begin_dt.normalize(),
                r.end_dt.normalize(),
                freq='D'
            )
        )

    if ranges:
        storm_dates = pd.DatetimeIndex(pd.Series(ranges).drop_duplicates())
        flights['storm_day_any_hawaii'] = (
            flights['FlightDate'].dt.normalize().isin(storm_dates)
        )

        overlap_aggs = {
            'flights': ('FlightDate', 'size'),
            'cancelled': ('Cancelled', 'sum'),
        }
        if 'ArrDelay' in flights.columns:
            overlap_aggs['mean_arr_delay'] = ('ArrDelay', 'mean')

        overlap = flights.groupby('storm_day_any_hawaii').agg(**overlap_aggs)
        overlap['cancel_rate_pct'] = (
            100 * overlap['cancelled'] / overlap['flights']
        )
        display(overlap.round(2))
    else:
        print('No valid storm date ranges were parsed.')
else:
    print('Storm datetime fields are missing; overlap deferred.')


## What to record after running

Keep the outputs for:

- class balance;
- yearly cancellation rates;
- airline summary;
- arrivals vs departures;
- weather shape, missingness, and candidate columns;
- storm-event type counts;
- broad storm-day overlap.

These results determine the exact merge strategy and baseline model design.
